In [ ]:
import paramiko
import re
import os
import time
from IPython.display import clear_output

# Paths & Cluster Config

Edit the paths below to match your cluster setup before running.

In [ ]:
# Cluster login
CLUSTER_HOST = 'loginserver.elsc.huji.ac.il'
USERNAME = 'qixin.yang'
SSH_KEY_PATH = os.path.expanduser('~/.ssh/id_rsa')  # set to None to use SSH agent

# Repo root on the cluster
PIPELINE_WORKDIR = '/ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline'
sh_script = f'{PIPELINE_WORKDIR}/utils/run_python_job.sh'

# Analysis paths on the cluster (inside the repo)
DATA_ROOT   = f'{PIPELINE_WORKDIR}/miniVI_PlaceCell_analysis_V4/data'
FIGURES_ROOT = f'{PIPELINE_WORKDIR}/miniVI_PlaceCell_analysis_V4/figures'
LOG_DIR     = f'{PIPELINE_WORKDIR}/miniVI_PlaceCell_analysis_V4/logs'

# Job parameters
N_JOBS       = 15        # parallel surrogate jobs (use number of CPUs - 1)
N_SURROGATES = 1000
DIRECTION_MODE = 'head'  # 'head' or 'travel'
FIRST_N_MINUTES = 10.0
FORCE_RECOMPUTE = False

JOB_NAME = f'egocentric_{DIRECTION_MODE}'
PYTHON_SCRIPT = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_analysis.py'

# Connect to Cluster

In [ ]:
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
kwargs = {}
if SSH_KEY_PATH and os.path.exists(SSH_KEY_PATH):
    kwargs['key_filename'] = SSH_KEY_PATH
ssh.connect(CLUSTER_HOST, username=USERNAME, **kwargs)
print('[INFO] Connected to cluster.')

def run_command(command):
    """Run a command on the cluster with a login shell."""
    stdin, stdout, stderr = ssh.exec_command(f"bash -l -c '{command}'")
    output = stdout.read().decode().strip()
    error  = stderr.read().decode().strip()
    return output, error

def wait_for_jobs(job_ids, poll_interval=60):
    """Poll SLURM until all jobs complete. Returns True if all succeeded."""
    if not job_ids:
        print('No jobs to wait for.')
        return True
    pending_jobs = set(job_ids)
    failed_jobs = []
    while pending_jobs:
        job_list = ','.join(pending_jobs)
        output, error = run_command(f'sacct -j {job_list} --format=JobID,State,ExitCode -n -P')
        if error and 'Invalid job id' not in error:
            print(f'[WARNING] {error}')
        completed = set()
        for line in output.strip().splitlines():
            if not line or '.' in line.split('|')[0]:
                continue
            parts = line.split('|')
            if len(parts) >= 2:
                job_id, state = parts[0], parts[1]
                if state in ['COMPLETED', 'FAILED', 'CANCELLED', 'TIMEOUT']:
                    completed.add(job_id)
                    if state != 'COMPLETED':
                        failed_jobs.append((job_id, state))
        pending_jobs -= completed
        clear_output(wait=True)
        print(f'[{time.strftime("%H:%M:%S")}] Jobs status:')
        print(f'  Completed: {len(job_ids) - len(pending_jobs)}/{len(job_ids)}')
        print(f'  Pending:   {len(pending_jobs)}')
        if failed_jobs:
            print(f'  Failed:    {failed_jobs}')
        if pending_jobs:
            print(f'\nWaiting {poll_interval}s before next check...')
            time.sleep(poll_interval)
    print('\n' + '=' * 50)
    if failed_jobs:
        print(f'WARNING: {len(failed_jobs)} job(s) failed: {failed_jobs}')
        return False
    print('All jobs completed successfully!')
    return True

# Step 0: Setup Data Links (run once)

Creates symlinks from `data/ANIMAL_NAME/merged_aligned_data.pkl` to the actual files on the HPC network.
Safe to re-run — existing links are skipped.

In [ ]:
setup_script = 'miniVI_PlaceCell_analysis_V4/notebooks_HPC/setup_data_links.py'
output, error = run_command(f'cd {PIPELINE_WORKDIR} && python {setup_script}')
if error:
    print(f'[ERROR]\n{error}')
print(output)

# Step 1: Submit Egocentric Analysis Job

In [ ]:
# Ensure log and figures dirs exist on the cluster
_, mkdir_err = run_command(f'mkdir -p {LOG_DIR} {FIGURES_ROOT}')
if mkdir_err:
    print(f'[WARN] mkdir: {mkdir_err}')

force_flag = '--force-recompute' if FORCE_RECOMPUTE else ''

cmd = (
    f'sbatch --job-name {JOB_NAME} --chdir {PIPELINE_WORKDIR} '
    f'--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err '
    f'{sh_script} {PIPELINE_WORKDIR} {PYTHON_SCRIPT} '
    f'--data-root {DATA_ROOT} '
    f'--figures-root {FIGURES_ROOT} '
    f'--n-jobs {N_JOBS} '
    f'--direction-mode {DIRECTION_MODE} '
    f'--n-surrogates {N_SURROGATES} '
    f'--first-n-minutes {FIRST_N_MINUTES} '
    f'{force_flag}'
)

output, error = run_command(cmd)
job_id = None
if error:
    print(f'[ERROR] Failed to submit job:\n{error}')
else:
    print(f'[INFO] {output}')
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        job_id = match.group(1)
        print(f'[INFO] Job ID: {job_id}')
        print(f'[INFO] Log: {LOG_DIR}/{JOB_NAME}_{job_id}.out')

# Step 2: Monitor Job (optional)

Polls SLURM every 60s. You can skip this and check manually with `sacct -j <job_id>`.

In [ ]:
if job_id:
    wait_for_jobs([job_id], poll_interval=60)
else:
    print('[INFO] No job_id found. Submit the job first.')

# Step 3: Check Job Log

In [ ]:
if job_id:
    log_path = f'{LOG_DIR}/{JOB_NAME}_{job_id}.out'
    output, error = run_command(f'tail -50 {log_path}')
    print(output)
    if error:
        print(f'[ERROR] {error}')
else:
    print('[INFO] No job_id defined.')